# Column Decisions & Save v2 Parquet

Apply the column keep/drop decisions from the review in notebook 02, then save a clean v2 parquet.

**Decisions summary:**

| Action | Columns |
|--------|---------|
| **KEEP** | `wy_player_id`, `wyscout_height`, `wyscout_weight`, `player_season_age` |
| **KEEP** | `tm_transfer_value`, `tm_transfer_fee` |
| **KEEP** | `from/to_Minutes`, `from/to_competition`, `from/to_position`, `from/to_season`, `from/to_team_id` |
| **KEEP** | `from/to_comp_division`, `from/to_comp_season_id`, `from/to_comp_season_name` |
| **KEEP** | All performance metrics (quality scores, per-90, z-scores) — both sides |
| **KEEP** | All team stats — both sides |
| **DROP** | `wyscout_birth_country`, `wyscout_first_name`, `wyscout_last_name`, `short_name`, `birth_date` |
| **DROP** | `wyscout_image_url`, `wyscout_passport`, `wyscout_role` |
| **DROP** | `wyscout_foot` (77.9% null — unusable) |
| **DROP** | `transfer_type`, `competition_start_date`, `first_played_date`, `last_played_date` |
| **DROP** | `tm_player_id`, `tm_transfer_date`, `tm_remaining_contract_days`, `tm_contract_until_date` |
| **DROP** | `from/to_comp_completed`, `from/to_comp_country`, `from/to_comp_name`, `from/to_comp_end_date`, `from/to_comp_start_date` |

In [1]:
import pandas as pd
import os
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

INPUT_PATH = "../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers.parquet"
OUTPUT_PATH = "../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v2.parquet"

df = pd.read_parquet(INPUT_PATH)
print(f"Input: {df.shape[0]:,} rows x {df.shape[1]} columns")

Input: 20,043 rows x 539 columns


## 1. Define columns to drop

In [2]:
DROP_COLUMNS = [
    # Identity — not useful for modeling
    "wyscout_birth_country",
    "wyscout_first_name",
    "wyscout_last_name",
    "short_name",
    "birth_date",
    "wyscout_image_url",
    "wyscout_passport",
    "wyscout_role",
    "wyscout_foot",  # 77.9% null

    # Transfer metadata — redundant or not features
    "transfer_type",
    "competition_start_date",
    "first_played_date",
    "last_played_date",
    "tm_player_id",
    "tm_transfer_date",
    "tm_remaining_contract_days",
    "tm_contract_until_date",

    # Competition metadata — redundant or captured elsewhere
    "from_comp_completed",
    "from_comp_country",
    "from_comp_name",
    "from_comp_end_date",
    "from_comp_start_date",
    "to_comp_completed",
    "to_comp_country",
    "to_comp_name",
    "to_comp_end_date",
    "to_comp_start_date",
]

# Verify all columns exist in the dataframe
missing = [c for c in DROP_COLUMNS if c not in df.columns]
if missing:
    print(f"WARNING: columns not found in df: {missing}")
else:
    print(f"All {len(DROP_COLUMNS)} columns to drop found in dataframe.")

All 27 columns to drop found in dataframe.


## 2. Drop & verify

In [3]:
df_v2 = df.drop(columns=DROP_COLUMNS)

print(f"Before: {df.shape[1]} columns")
print(f"Dropped: {len(DROP_COLUMNS)} columns")
print(f"After:  {df_v2.shape[1]} columns")
print(f"Rows:   {df_v2.shape[0]:,} (unchanged)")
print()

# Quick sanity: show remaining column groups
remaining = df_v2.columns.tolist()
groups = {
    "player_meta": [c for c in remaining if c in ["wy_player_id", "wyscout_height", "wyscout_weight", "player_season_age"]],
    "transfer_financials": [c for c in remaining if c.startswith("tm_")],
    "structural_from": [c for c in remaining if c in ["from_Minutes", "from_competition", "from_position", "from_season", "from_team_id"]],
    "structural_to": [c for c in remaining if c in ["to_Minutes", "to_competition", "to_position", "to_season", "to_team_id"]],
    "comp_from": [c for c in remaining if c.startswith("from_comp_")],
    "comp_to": [c for c in remaining if c.startswith("to_comp_")],
    "perf_from (quality+per90)": [c for c in remaining if c.startswith("from_") and not c.startswith("from_z_score_")
                                   and not c.startswith("from_team_stats_") and not c.startswith("from_comp_")
                                   and c not in ["from_Minutes", "from_competition", "from_position", "from_season", "from_team_id"]],
    "z_scores_from": [c for c in remaining if c.startswith("from_z_score_")],
    "team_stats_from": [c for c in remaining if c.startswith("from_team_stats_")],
    "perf_to (quality+per90)": [c for c in remaining if c.startswith("to_") and not c.startswith("to_z_score_")
                                 and not c.startswith("to_team_stats_") and not c.startswith("to_comp_")
                                 and c not in ["to_Minutes", "to_competition", "to_position", "to_season", "to_team_id"]],
    "z_scores_to": [c for c in remaining if c.startswith("to_z_score_")],
    "team_stats_to": [c for c in remaining if c.startswith("to_team_stats_")],
}

print("Remaining column groups:")
total = 0
for name, cols in groups.items():
    print(f"  {name:30s} → {len(cols):3d}")
    total += len(cols)
print(f"  {'TOTAL':30s} → {total:3d}")

Before: 539 columns
Dropped: 27 columns
After:  512 columns
Rows:   20,043 (unchanged)

Remaining column groups:
  player_meta                    →   4
  transfer_financials            →   2
  structural_from                →   5
  structural_to                  →   5
  comp_from                      →   3
  comp_to                        →   3
  perf_from (quality+per90)      →  96
  z_scores_from                  →  75
  team_stats_from                →  74
  perf_to (quality+per90)        →  96
  z_scores_to                    →  75
  team_stats_to                  →  74
  TOTAL                          → 512


## 3. Save v2 parquet

In [4]:
df_v2.to_parquet(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved: {OUTPUT_PATH}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {df_v2.shape[0]:,} rows x {df_v2.shape[1]} columns")

Saved: ../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v2.parquet
Size:  43.4 MB
Shape: 20,043 rows x 512 columns
